In [4]:
# ============================================================
# TUGAS MANIPULASI DATA PANDAS - PAKET A TRANSAKSI POS 2024-2025
# ============================================================
import os
import glob
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)

# ------------------------------------------------------------
# PERSIAPAN: Mencari dan membaca file CSV secara otomatis
# ------------------------------------------------------------
print("Folder kerja notebook:", os.getcwd())

kandidat = glob.glob("paket_a*.csv")
if not kandidat:
    # Jika tidak ada di folder notebook, cari ke seluruh folder pengguna
    home = os.path.expanduser("~")
    kandidat = glob.glob(os.path.join(home, "**", "paket_a*.csv"), recursive=True)

if not kandidat:
    raise FileNotFoundError("File paket_a*.csv tidak ditemukan. Letakkan CSV di folder yang sama dengan notebook.")

nama_file = kandidat[0]
print("Membaca file:", nama_file)

df = pd.read_csv(nama_file)

# Merapikan spasi tersembunyi pada nama kolom dan kolom teks penting
df.columns = df.columns.str.strip()
df['status'] = df['status'].astype(str).str.strip().str.upper()
df['kode_gerai'] = df['kode_gerai'].astype(str).str.strip().str.upper()

print("\n=== INFO AWAL DATA ===")
print("Ukuran data (baris, kolom):", df.shape)
print("\nTipe data awal:")
print(df.dtypes)
print("\nJumlah nilai kosong per kolom:")
print(df.isna().sum())
print("\n5 baris pertama:")
print(df.head())

# ============================================================
# TASK 1 - INDEXING & SELECTION (Pembersihan Anomali & Imputasi)
# ============================================================
print("\n" + "=" * 60)
print("TASK 1")
print("=" * 60)

# 1. Pembersihan format harga
harga = df['harga_satuan'].astype(str).str.strip()

# Format ribuan bertitik (contoh '13.000' = 13000) dibersihkan lebih dulu
# agar tidak terbaca sebagai 13.0 saat dikonversi
mask_ribuan = harga.str.fullmatch(r'\d{1,3}(\.\d{3})+')
print("Jumlah harga berformat ribuan bertitik:", mask_ribuan.sum())
print("Jumlah harga berformat koma desimal   :", harga.str.contains(',').sum())
harga = harga.where(~mask_ribuan, harga.str.replace('.', '', regex=False))

# Koma desimal -> titik, lalu ubah ke float
harga = harga.str.replace(',', '.', regex=False)
df['harga_satuan'] = pd.to_numeric(harga).astype(float)
print("Tipe data harga_satuan sekarang:", df['harga_satuan'].dtype)

# Pastikan kolom numerik lain bertipe angka
df['jumlah'] = pd.to_numeric(df['jumlah'])
df['diskon_persen'] = pd.to_numeric(df['diskon_persen'])

# 2. Kuantitas negatif -> positif (Boolean Indexing + .loc[] + .abs())
print("\nJumlah negatif sebelum:", (df['jumlah'] < 0).sum())
df.loc[df['jumlah'] < 0, 'jumlah'] = df.loc[df['jumlah'] < 0, 'jumlah'].abs()
print("Jumlah negatif sesudah:", (df['jumlah'] < 0).sum())

# 3. Imputasi kode_kasir kosong dengan 'K_UNKNOWN'
print("\nNaN kode_kasir sebelum:", df['kode_kasir'].isna().sum())
df.loc[df['kode_kasir'].isna(), 'kode_kasir'] = 'K_UNKNOWN'
print("NaN kode_kasir sesudah:", df['kode_kasir'].isna().sum())

# 4. Seleksi posisi dengan .iloc[] -> id_transaksi, tanggal, kode_gerai, jumlah
print("\n5 baris pertama (id_transaksi, tanggal, kode_gerai, jumlah):")
print(df.iloc[:5, [0, 1, 2, 5]])

# ============================================================
# TASK 2 - OPERASI ARITMATIKA (Feature Engineering)
# ============================================================
print("\n" + "=" * 60)
print("TASK 2")
print("=" * 60)

# 1. Omzet kotor
df['Total_Kotor'] = df['harga_satuan'] * df['jumlah']

# 2. Nominal diskon (diskon_persen skala 0-100)
df['Nominal_Diskon'] = df['Total_Kotor'] * (df['diskon_persen'] / 100)

# 3. Omzet bersih + PPN 11%
df['Total_Bersih'] = (df['Total_Kotor'] - df['Nominal_Diskon']) * 1.11

print(df[['id_transaksi', 'harga_satuan', 'jumlah', 'diskon_persen',
          'Total_Kotor', 'Nominal_Diskon', 'Total_Bersih']].head())

# ============================================================
# TASK 3 - FILTERING
# ============================================================
print("\n" + "=" * 60)
print("TASK 3")
print("=" * 60)

# 1. Filter multi-kondisi
filter_multi = df[
    (df['status'] == 'SELESAI') &
    (df['kode_gerai'].isin(['G05', 'G09'])) &
    (df['Total_Bersih'] > 100000)
]
print("Jumlah baris yang memenuhi kriteria:", len(filter_multi))
print(filter_multi.head())

# 2. Filter dengan .query() -> gerai G10, jumlah minimal 5 unit
filter_g10 = df.query("kode_gerai == 'G10' and jumlah >= 5")
print("\nJumlah transaksi G10 dengan jumlah >= 5:", len(filter_g10))
print(filter_g10.head())

# ============================================================
# TASK 4 - SORTING
# ============================================================
print("\n" + "=" * 60)
print("TASK 4")
print("=" * 60)

# 1. Urutkan kode_gerai A-Z, lalu Total_Bersih tertinggi ke terendah
df_urut = df.sort_values(by=['kode_gerai', 'Total_Bersih'], ascending=[True, False])
print(df_urut[['id_transaksi', 'kode_gerai', 'Total_Bersih']].head(8))

# 2. Top-5 transaksi dengan Total_Bersih tertinggi
top5 = df.nlargest(5, 'Total_Bersih')
print("\nTop 5 transaksi Total_Bersih tertinggi:")
print(top5[['id_transaksi', 'kode_gerai', 'jumlah', 'Total_Bersih']])

# ============================================================
# TASK 5 - GROUPING & AGREGASI
# ============================================================
print("\n" + "=" * 60)
print("TASK 5")
print("=" * 60)

# 1. Laporan agregasi per gerai (hanya status SELESAI)
df_selesai = df[df['status'] == 'SELESAI'].copy()

laporan_gerai = (
    df_selesai.groupby('kode_gerai')
    .agg(
        Jumlah_Transaksi=('id_transaksi', 'count'),
        Total_Unit_Terjual=('jumlah', 'sum'),
        Total_Omzet_Bersih=('Total_Bersih', 'sum'),
        Rata_Rata_Transaksi=('Total_Bersih', 'mean')
    )
    .sort_values('Total_Omzet_Bersih', ascending=False)
    .round(2)
)
print("Laporan Kinerja Gerai:")
print(laporan_gerai)

# 2. Transformasi kelompok (window function) dengan groupby().transform()
df['Kontribusi_Gerai_Persen'] = (
    df['Total_Bersih'] / df.groupby('kode_gerai')['Total_Bersih'].transform('sum')
) * 100

print("\nKontribusi per transaksi terhadap omzet gerainya:")
print(df[['id_transaksi', 'kode_gerai', 'Total_Bersih', 'Kontribusi_Gerai_Persen']].head())

# Verifikasi: total kontribusi tiap gerai harus 100
print("\nVerifikasi total kontribusi per gerai (harus 100):")
print(df.groupby('kode_gerai')['Kontribusi_Gerai_Persen'].sum().round(2))

# ------------------------------------------------------------
# (Opsional) Simpan hasil ke file CSV
# ------------------------------------------------------------
# df.to_csv("hasil_transaksi_bersih_paket_a.csv", index=False)
# laporan_gerai.to_csv("laporan_gerai_paket_a.csv")

Folder kerja notebook: C:\Users\Rendy Ramadhan\tugas_pandas
Membaca file: paket_a_transaksi_pos_2024_2025.csv

=== INFO AWAL DATA ===
Ukuran data (baris, kolom): (5030, 9)

Tipe data awal:
id_transaksi       str
tanggal            str
kode_gerai         str
kode_kasir         str
kode_produk        str
jumlah           int64
harga_satuan       str
diskon_persen    int64
status             str
dtype: object

Jumlah nilai kosong per kolom:
id_transaksi       0
tanggal            0
kode_gerai         0
kode_kasir       261
kode_produk        0
jumlah             0
harga_satuan       0
diskon_persen      0
status             0
dtype: int64

5 baris pertama:
  id_transaksi     tanggal kode_gerai kode_kasir kode_produk  jumlah harga_satuan  diskon_persen   status
0   TRX0241065  2025-06-02        G09       K011     RMT1402       4        13000              0  SELESAI
1   TRX0240510  2024-12-28        G09       K015     SGR1205       1        21000              0  SELESAI
2   TRX0244483  2024